# 02 — Company split and EDA

**Purpose.** Create the company-level outer train/test split and produce the descriptive tables and supporting EDA figure.

**Inputs.** `data/processed/model_dataset.csv` from Notebook 01.

**Outputs.** `train.csv`, `test.csv`, split summary, annual failure rates, year distributions, class-feature summaries, and an EDA figure.

**What you will learn.** Why the split is done by company rather than by row, and how the training and final-test samples compare.

**Run first.** Run Notebook 01 first.

## Imports and paths

In [1]:
from __future__ import annotations

import ast
import json
import math
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

current_path = Path.cwd().resolve()
if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

assert project_root.name == "ml-finance-bankruptcy-analysis", (
    f"Unexpected project root: {project_root}"
)

DATA_DIR = project_root / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = project_root / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
REPORT_DIR = OUTPUTS_DIR / "report"
PAPER_TABLES_DIR = OUTPUTS_DIR / "paper" / "tables"
PAPER_FIGURES_DIR = OUTPUTS_DIR / "paper" / "figures"

for path in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    REPORT_DIR,
    PAPER_TABLES_DIR,
    PAPER_FIGURES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)


## Project constants

In [2]:
RAW_DATA_PATH = RAW_DATA_DIR / "american_bankruptcy.csv"
MODEL_DATASET_PATH = PROCESSED_DATA_DIR / "model_dataset.csv"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "test.csv"

COMPANY_COLUMN = "company_name"
RAW_TARGET_COLUMN = "status_label"
TARGET_COLUMN = "failed"
YEAR_COLUMN = "year"

ALIVE_LABEL = "alive"
FAILED_LABEL = "failed"

RANDOM_STATE = 42
OUTER_TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
LOGISTIC_C_GRID = [0.01, 0.1, 1.0, 10.0]
PCA_COMPONENT_GRID = [2, 3, 5, 8, 10, 12, 15, 18]

EXPECTED_MODEL_ORDER = [
    "Majority-class baseline",
    "Logistic Regression",
    "L1 Regularized Logistic Regression",
    "L2 Regularized Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
]

PREDICTION_TABLE_COLUMNS = [
    "model",
    COMPANY_COLUMN,
    YEAR_COLUMN,
    "actual_failed",
    "predicted_failed",
    "probability_failed",
]
PREDICTION_PROBABILITY_FLOAT_FORMAT = "%.11f"


## Feature names and descriptions

In [3]:
FEATURE_NAME_MAP = {
    "X1": "Current assets",
    "X2": "Cost of goods sold",
    "X3": "Depreciation and amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net income",
    "X7": "Total receivables",
    "X8": "Market value",
    "X9": "Net sales",
    "X10": "Total assets",
    "X11": "Total long-term debt",
    "X12": "EBIT",
    "X13": "Gross profit",
    "X14": "Total current liabilities",
    "X15": "Retained earnings",
    "X16": "Total revenue",
    "X17": "Total liabilities",
    "X18": "Total operating expenses",
}

FEATURE_DESCRIPTION_MAP = {
    "X1": "Assets expected to be sold, converted into cash, or used within one year.",
    "X2": "Direct costs related to producing or selling the firm's goods and services.",
    "X3": "Depreciation of tangible assets and amortization of intangible assets.",
    "X4": "Earnings before interest, taxes, depreciation, and amortization.",
    "X5": "Goods and raw materials held by the firm for production or sale.",
    "X6": "Profit after expenses and costs have been deducted from revenue.",
    "X7": "Money owed to the firm by customers for delivered goods or services.",
    "X8": "Market capitalization or market value of the publicly traded company.",
    "X9": "Gross sales minus returns, allowances, and discounts.",
    "X10": "Total assets owned or controlled by the company.",
    "X11": "Debt obligations due after more than one year.",
    "X12": "Earnings before interest and taxes.",
    "X13": "Profit after subtracting costs related to manufacturing and selling.",
    "X14": "Short-term obligations due within one year.",
    "X15": "Accumulated profits retained in the business after dividends and losses.",
    "X16": "Total income from sales before subtracting expenses.",
    "X17": "Total debts and obligations owed to outside parties.",
    "X18": "Expenses incurred through normal business operations.",
}

KEY_FEATURES_FOR_EDA = ["X8", "X6", "X11", "X1", "X17", "X15"]
KEY_MODELS_FOR_THRESHOLD_ANALYSIS = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting",
]


## Figure style helpers

In [4]:
MODEL_COLORS = {
    "Majority-class baseline": "#8c8c8c",
    "Logistic Regression": "#4c78a8",
    "Random Forest": "#59a14f",
    "Gradient Boosting": "#f28e2b",
}
OUTCOME_COLORS = {"detected": "#59a14f", "missed": "#e15759", "false_alarm": "#f28e2b"}
DIRECTION_COLORS = {"positive": "#c44e52", "negative": "#4c78a8"}
METRIC_COLORS = {
    "Precision": "#4c78a8",
    "Recall": "#e15759",
    "F1": "#59a14f",
    "ROC-AUC": "#4c78a8",
    "PR-AUC": "#f28e2b",
    "Failed F1": "#59a14f",
}
BASELINE_COLOR = "#8c8c8c"


def apply_project_style() -> None:
    """Apply the common Matplotlib style used across project figures."""
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.titlesize": 12,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.8,
            "lines.linewidth": 2.0,
            "lines.markersize": 5.5,
        }
    )


def style_axis(ax, *, grid_axis: str = "y", percent_y: bool = False, integer_x: bool = False) -> None:
    """Add consistent grid and tick formatting to one axis."""
    ax.grid(axis=grid_axis, linestyle="--", alpha=0.25)
    if percent_y:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    if integer_x:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def save_figure(fig, output_path: Path) -> None:
    """Save a Matplotlib figure with the project's standard export settings."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


## Data validation and feature helpers

In [5]:
def validate_required_columns(data: pd.DataFrame) -> None:
    """Confirm that the raw dataset has the identifier, year, and target columns."""
    missing = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")


def validate_target_labels(data: pd.DataFrame) -> None:
    """Confirm that the raw target contains only the expected alive/failed labels."""
    observed = set(data[RAW_TARGET_COLUMN].dropna().unique())
    unexpected = observed.difference({ALIVE_LABEL, FAILED_LABEL})
    if unexpected:
        raise ValueError(f"Unexpected target labels found: {sorted(unexpected)}")


def identify_feature_columns(data: pd.DataFrame) -> list[str]:
    """Return numeric financial variables from the raw dataset, excluding identifiers."""
    excluded = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}
    return [
        column
        for column in data.columns
        if column not in excluded and pd.api.types.is_numeric_dtype(data[column])
    ]


def get_feature_columns(data: pd.DataFrame, include_year: bool = False) -> list[str]:
    """Return modelling predictors while keeping identifiers and target out of X."""
    excluded = {COMPANY_COLUMN, TARGET_COLUMN}
    if not include_year:
        excluded.add(YEAR_COLUMN)
    return [column for column in data.columns if column not in excluded]


def split_features_target(data: pd.DataFrame, include_year: bool = False) -> tuple[pd.DataFrame, pd.Series]:
    """Split a model-ready table into predictor matrix and binary target."""
    feature_columns = get_feature_columns(data, include_year=include_year)
    return data[feature_columns].copy(), data[TARGET_COLUMN].copy()


## Company split helpers

In [6]:
def create_company_level_split(
    data: pd.DataFrame,
    test_size: float = OUTER_TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split rows by company so a firm cannot appear in both samples."""
    missing = {COMPANY_COLUMN, TARGET_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required split columns: {sorted(missing)}")

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(
        splitter.split(data, y=data[TARGET_COLUMN], groups=data[COMPANY_COLUMN])
    )
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()


def check_no_company_overlap(left: pd.DataFrame, right: pd.DataFrame) -> bool:
    """Return True when two samples have disjoint company identifiers."""
    left_companies = set(left[COMPANY_COLUMN].unique())
    right_companies = set(right[COMPANY_COLUMN].unique())
    return left_companies.isdisjoint(right_companies)


def create_split_summary(
    full_data: pd.DataFrame,
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize row counts, company counts, and failure rates for a split."""
    def summarize(name: str, data: pd.DataFrame) -> dict[str, float | int | str]:
        return {
            "split": name,
            "n_rows": int(len(data)),
            "n_companies": int(data[COMPANY_COLUMN].nunique()),
            "n_failed_rows": int(data[TARGET_COLUMN].sum()),
            "n_alive_rows": int(len(data) - data[TARGET_COLUMN].sum()),
            "failure_rate": float(data[TARGET_COLUMN].mean()),
        }

    summary = pd.DataFrame(
        [summarize("full", full_data), summarize("train", train_data), summarize("test", test_data)]
    )
    summary["company_share"] = summary["n_companies"] / int(full_data[COMPANY_COLUMN].nunique())
    summary["row_share"] = summary["n_rows"] / int(len(full_data))
    return summary


## Evaluation helpers

In [7]:
def get_probability_failed(model, x: pd.DataFrame) -> np.ndarray:
    """Return predicted probabilities for the failed class."""
    return model.predict_proba(x)[:, 1]


def evaluate_binary_classifier(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> dict[str, float | int | str]:
    """Calculate the common classification metrics for one fitted model."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, probability_failed),
        "pr_auc": average_precision_score(y_true, probability_failed),
        "precision_failed": precision_score(y_true, y_pred, zero_division=0),
        "recall_failed": recall_score(y_true, y_pred, zero_division=0),
        "f1_failed": f1_score(y_true, y_pred, zero_division=0),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


def create_classification_report_table(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
) -> pd.DataFrame:
    """Convert scikit-learn's classification report into a flat table."""
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["alive", "failed"],
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            rows.append({"model": model_name, "class_or_average": label, **metrics})
    return pd.DataFrame(rows)


def create_prediction_table(
    data: pd.DataFrame,
    model_name: str,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> pd.DataFrame:
    """Create an identifier-preserving prediction table for one model."""
    table = pd.DataFrame(
        {
            "model": model_name,
            COMPANY_COLUMN: data[COMPANY_COLUMN].to_numpy(),
            YEAR_COLUMN: data[YEAR_COLUMN].to_numpy(),
            "actual_failed": data[TARGET_COLUMN].to_numpy(),
            "predicted_failed": y_pred,
            "probability_failed": probability_failed,
        }
    )
    return table[PREDICTION_TABLE_COLUMNS]


def save_prediction_table(predictions: pd.DataFrame, output_path: Path) -> None:
    """Write prediction probabilities with deterministic decimal formatting."""
    predictions[PREDICTION_TABLE_COLUMNS].to_csv(
        output_path,
        index=False,
        float_format=PREDICTION_PROBABILITY_FLOAT_FORMAT,
    )


## Model builders and selection helpers

In [8]:
def build_majority_class_baseline() -> DummyClassifier:
    """Build the benchmark model that always predicts the most frequent class."""
    return DummyClassifier(strategy="most_frequent")


def build_logistic_pipeline(
    penalty: str = "l2",
    c_value: float = 1.0,
    class_weight: str | dict | None = "balanced",
) -> Pipeline:
    """Build a scaled Logistic Regression pipeline with the original settings."""
    if penalty not in {"l1", "l2"}:
        raise ValueError("Only 'l1' and 'l2' penalties are supported in this project.")

    l1_ratio = 1.0 if penalty == "l1" else 0.0
    model = LogisticRegression(
        C=c_value,
        l1_ratio=l1_ratio,
        solver="saga",
        class_weight=class_weight,
        max_iter=5_000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(steps=[("scaler", StandardScaler()), ("model", model)])


def build_interpretable_logit() -> Pipeline:
    """Build the fixed Logistic Regression benchmark used for interpretation."""
    return build_logistic_pipeline(penalty="l2", c_value=1.0)


def select_regularized_logit(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
    penalty: str,
    c_grid: list[float],
) -> tuple[Pipeline, float, float]:
    """Select L1 or L2 Logistic Regression by validation PR-AUC."""
    best_model, best_c, best_score = None, None, -1.0
    for c_value in c_grid:
        candidate = build_logistic_pipeline(penalty=penalty, c_value=c_value)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model, best_c, best_score = candidate, c_value, score
    if best_model is None or best_c is None:
        raise RuntimeError("No Logistic Regression model was selected.")
    return best_model, best_c, best_score


def build_decision_tree(max_depth: int | None = 3, min_samples_leaf: int = 100) -> DecisionTreeClassifier:
    """Build a class-weighted Decision Tree with the source project settings."""
    return DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def build_random_forest(
    n_estimators: int = 300,
    max_depth: int | None = 5,
    min_samples_leaf: int = 50,
    max_features: str | None = "sqrt",
) -> RandomForestClassifier:
    """Build a Random Forest using the original class weights and seed."""
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def build_gradient_boosting(
    n_estimators: int = 150,
    learning_rate: float = 0.05,
    max_depth: int = 2,
    min_samples_leaf: int = 100,
) -> GradientBoostingClassifier:
    """Build a Gradient Boosting classifier with the source project settings."""
    return GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
    )


def select_decision_tree(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[DecisionTreeClassifier, dict[str, int], float]:
    """Search the original Decision Tree grid and keep the best PR-AUC model."""
    grid = {"max_depth": [2, 3, 4, 5], "min_samples_leaf": [50, 100, 200]}
    best_model, best_params, best_score = None, None, -1.0
    for max_depth, min_samples_leaf in product(grid["max_depth"], grid["min_samples_leaf"]):
        candidate = build_decision_tree(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf}
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Decision Tree model was selected.")
    return best_model, best_params, best_score


def select_random_forest(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[RandomForestClassifier, dict[str, object], float]:
    """Search the original Random Forest grid and keep the best PR-AUC model."""
    grid = {
        "n_estimators": [300],
        "max_depth": [4, 6, None],
        "min_samples_leaf": [50, 100],
        "max_features": ["sqrt"],
    }
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, max_depth, min_samples_leaf, max_features in product(
        grid["n_estimators"], grid["max_depth"], grid["min_samples_leaf"], grid["max_features"]
    ):
        candidate = build_random_forest(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
        )
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "max_features": max_features,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Random Forest model was selected.")
    return best_model, best_params, best_score


def select_gradient_boosting(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[GradientBoostingClassifier, dict[str, object], float]:
    """Search the original Gradient Boosting grid using balanced sample weights."""
    grid = {
        "n_estimators": [100, 150],
        "learning_rate": [0.03, 0.05],
        "max_depth": [2, 3],
        "min_samples_leaf": [100],
    }
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, learning_rate, max_depth, min_samples_leaf in product(
        grid["n_estimators"], grid["learning_rate"], grid["max_depth"], grid["min_samples_leaf"]
    ):
        candidate = build_gradient_boosting(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
        )
        candidate.fit(x_train, y_train, sample_weight=sample_weight)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Gradient Boosting model was selected.")
    return best_model, best_params, best_score


## Model reconstruction helpers

In [9]:
def parse_selected_parameters(value: object) -> dict[str, object]:
    """Parse the saved parameter string from `model_specification.csv`."""
    if pd.isna(value) or value == "":
        return {}

    text = str(value)
    if text.startswith("{"):
        return ast.literal_eval(text)

    params: dict[str, object] = {}
    for piece in text.split(","):
        if "=" in piece:
            key, val = piece.strip().split("=", 1)
            params[key] = float(val) if key == "C" else val
    return params


def build_model_from_spec(model_name: str, selected_parameters: object):
    """Recreate one model from its saved specification without using model binaries."""
    params = parse_selected_parameters(selected_parameters)
    if model_name == "Majority-class baseline":
        return build_majority_class_baseline()
    if model_name == "Logistic Regression":
        return build_interpretable_logit()
    if model_name == "L1 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l1", c_value=params["C"])
    if model_name == "L2 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l2", c_value=params["C"])
    if model_name == "Decision Tree":
        return build_decision_tree(**params)
    if model_name == "Random Forest":
        return build_random_forest(**params)
    if model_name == "Gradient Boosting":
        return build_gradient_boosting(**params)
    raise ValueError(f"Unknown model: {model_name}")


## Company-Level Outer Split

In [10]:

model_dataset = pd.read_csv(MODEL_DATASET_PATH)
train_data, test_data = create_company_level_split(model_dataset, test_size=OUTER_TEST_SIZE, random_state=RANDOM_STATE)
assert check_no_company_overlap(train_data, test_data), 'Outer split contains company overlap.'
assert len(train_data) + len(test_data) == len(model_dataset), 'Outer split must preserve all rows.'
assert set(train_data[TARGET_COLUMN].unique()) == {0, 1}, 'Train split must contain both target classes.'
assert set(test_data[TARGET_COLUMN].unique()) == {0, 1}, 'Test split must contain both target classes.'
split_summary = create_split_summary(model_dataset, train_data, test_data)
assert set(split_summary['split']) == {'full', 'train', 'test'}, 'Split summary must contain full, train, and test.'
assert math.isclose(split_summary['row_share'].sum(), 2.0), 'Full, train, and test row shares should sum to 2.'
train_data.to_csv(TRAIN_DATA_PATH, index=False)
test_data.to_csv(TEST_DATA_PATH, index=False)
split_summary.to_csv(TABLES_DIR / 'split_summary.csv', index=False)
split_summary


,split,n_rows,n_companies,n_failed_rows,n_alive_rows,failure_rate,company_share,row_share
0,full,78682,8971,5220,73462,0.066343,1.000000,1.000000
1,train,62988,7176,4043,58945,0.064187,0.799911,0.800539
2,test,15694,1795,1177,14517,0.074997,0.200089,0.199461


## Exploratory Summaries

In [11]:

annual_failure_rate = model_dataset.groupby(YEAR_COLUMN)[TARGET_COLUMN].agg(n_observations='count', n_failed='sum').reset_index()
annual_failure_rate['n_alive'] = annual_failure_rate['n_observations'] - annual_failure_rate['n_failed']
annual_failure_rate['failure_rate'] = annual_failure_rate['n_failed'] / annual_failure_rate['n_observations']
train_years = train_data.groupby(YEAR_COLUMN).size().reset_index(name='n_observations').assign(split='train')
test_years = test_data.groupby(YEAR_COLUMN).size().reset_index(name='n_observations').assign(split='test')
year_distribution = pd.concat([train_years, test_years], ignore_index=True)
year_distribution['share_within_split'] = year_distribution['n_observations'] / year_distribution.groupby('split')['n_observations'].transform('sum')
year_distribution = year_distribution[['split', YEAR_COLUMN, 'n_observations', 'share_within_split']]
rows = []
for feature in KEY_FEATURES_FOR_EDA:
    for status_value, status_name in [(0, 'alive'), (1, 'failed')]:
        subset = model_dataset.loc[model_dataset[TARGET_COLUMN] == status_value, feature]
        rows.append({'feature': feature, 'readable_name': FEATURE_NAME_MAP.get(feature, feature), 'status': status_name, 'mean': subset.mean(), 'median': subset.median(), 'std': subset.std(), 'n_observations': int(subset.shape[0])})
class_feature_summary = pd.DataFrame(rows)
annual_failure_rate.to_csv(TABLES_DIR / 'annual_failure_rate.csv', index=False)
year_distribution.to_csv(TABLES_DIR / 'train_test_year_distribution.csv', index=False)
class_feature_summary.to_csv(TABLES_DIR / 'class_feature_summary.csv', index=False)
annual_failure_rate.head()


,year,n_observations,n_failed,n_alive,failure_rate
0,1999,5308,380,4928,0.071590
1,2000,5226,404,4822,0.077306
2,2001,4897,414,4483,0.084542
3,2002,4651,414,4237,0.089013
4,2003,4417,415,4002,0.093955


## Key Feature Median Figure

In [12]:

medians = class_feature_summary.pivot(index=['feature', 'readable_name'], columns='status', values='median')
plot_data = medians.copy()
relative_scale = (plot_data['failed'].abs() + plot_data['alive'].abs()) / 2
plot_data['relative_median_difference'] = (plot_data['failed'] - plot_data['alive']) / relative_scale.replace(0, pd.NA)
plot_data = plot_data.reset_index()
plot_data['readable_name'] = pd.Categorical(plot_data['readable_name'], categories=[FEATURE_NAME_MAP.get(feature, feature) for feature in KEY_FEATURES_FOR_EDA], ordered=True)
plot_data = plot_data.sort_values('readable_name')
apply_project_style()
fig, ax = plt.subplots(figsize=(8.2, 4.8))
colors = ['#c44e52' if value > 0 else '#4c78a8' for value in plot_data['relative_median_difference']]
bars = ax.barh(plot_data['readable_name'], plot_data['relative_median_difference'], color=colors)
max_abs_difference = plot_data['relative_median_difference'].abs().max()
label_offset = max_abs_difference * 0.04
ax.set_title('Relative Median Gap by Bankruptcy Status', pad=22)
ax.text(0.5, 1.01, 'Descriptive comparison only; values do not imply causality.', transform=ax.transAxes, ha='center', va='bottom', fontsize=8.5, color='#4d4d4d')
ax.set_xlabel('Difference in group medians divided by average absolute magnitude')
ax.set_ylabel('Financial variable')
ax.axvline(0, linewidth=1, color='black')
ax.set_xlim(-max_abs_difference * 1.25, max_abs_difference * 1.25)
style_axis(ax, grid_axis='x')
ax.xaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
for bar, value in zip(bars, plot_data['relative_median_difference'], strict=False):
    ax.text(value + (label_offset if value >= 0 else -label_offset), bar.get_y() + bar.get_height() / 2, f'{value:.0%}', va='center', ha='left' if value >= 0 else 'right', fontsize=8)
save_figure(fig, FIGURES_DIR / 'key_feature_median_by_status.png')
assert (FIGURES_DIR / 'key_feature_median_by_status.png').exists(), 'EDA figure was not saved.'
class_feature_summary


,feature,readable_name,status,mean,median,std,n_observations
0,X8,Market value,alive,3596.015157,240.89525,19022.527429,73462
1,X8,Market value,failed,857.813011,117.79940,3396.453790,5220
2,X6,Net income,alive,141.994726,2.07150,1299.259556,73462
3,X6,Net income,failed,-48.112333,-3.32700,592.050918,5220
4,X11,Total long-term debt,alive,730.516984,7.09250,3309.223955,73462
5,X11,Total long-term debt,failed,609.430007,18.43850,2077.661309,5220
6,X1,Current assets,alive,914.542615,102.91750,4052.047889,73462
7,X1,Current assets,failed,399.339353,75.87150,1147.837282,5220
8,X17,Total liabilities,alive,1809.571974,80.74050,8257.442726,73462
9,X17,Total liabilities,failed,1266.816748,97.03500,4221.179588,5220
